# Notebook 03 — Extracción de Embeddings

**Prerequisito:** `artifacts/checkpoints/encoder_best.pt` debe existir (entrenado en notebook 02 o en Colab).

Este notebook genera los embeddings de 1024 dimensiones para train/val/test
y los guarda en `data/embeddings/`. Todos los modelos clásicos usarán exactamente estos embeddings.

## Cómo ejecutar
```bash
jupyter lab notebooks/03_extract_embeddings.ipynb
# O directamente:
python -m scripts.extract_embeddings --checkpoint artifacts/checkpoints/encoder_best.pt
```

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

# COLAB — descomenta si ejecutas en Colab
# !git clone https://github.com/TU_USUARIO/Malaria-Dectetion-Deeplearning.git
# %cd Malaria-Dectetion-Deeplearning
# !pip install -r requirements.txt -q

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

from src.utils.seed import set_global_seed
from src.utils.io import load_config, load_checkpoint, save_embeddings
from src.data.augmentations import get_eval_transform
from src.data.dataset import MalariaDataset
from src.models.encoder import ContrastiveEncoder

set_global_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

In [ ]:
CHECKPOINT_PATH = 'artifacts/checkpoints/encoder_best.pt'
assert Path(CHECKPOINT_PATH).exists(), f'No existe {CHECKPOINT_PATH}. Entrena primero con notebook 02.'

cfg = load_config('configs/contrastive.yaml')
data_cfg = load_config('configs/data.yaml')

enc_cfg = cfg.get('encoder', {})
model = ContrastiveEncoder(
    embedding_dim=enc_cfg.get('embedding_dim', 1024),
    proj_dim=enc_cfg.get('proj_dim', 128),
    pretrained=False,
).to(device)

ckpt = load_checkpoint(CHECKPOINT_PATH, device=str(device))
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Checkpoint cargado — epoch {ckpt.get("epoch", "?")} | val_loss={ckpt.get("val_loss", "?"): .4f}')

In [ ]:
transform = get_eval_transform(img_size=data_cfg.get('img_size', 96))
processed_dir = Path(data_cfg['processed_dir'])
embeddings_dir = Path(data_cfg['embeddings_dir'])
embeddings_dir.mkdir(parents=True, exist_ok=True)

for split in ['train', 'val', 'test']:
    ds = MalariaDataset(processed_dir / f'{split}.csv', transform=transform)
    loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=0)
    
    all_emb, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=f'  {split}'):
            emb = model(imgs.to(device), return_embedding=True).cpu().numpy()
            all_emb.append(emb)
            all_labels.append(labels.numpy())
    
    X = np.concatenate(all_emb)
    y = np.concatenate(all_labels)
    assert X.shape[1] == 1024, f'Error: embedding dim={X.shape[1]}'
    save_embeddings(X, y, split, embeddings_dir)
    print(f'{split}: {X.shape} | pos={y.sum()}/{len(y)}')

print('\nEmbeddings guardados en', embeddings_dir.resolve())

In [ ]:
# Verificación rápida
from src.utils.io import load_embeddings
X_train, y_train = load_embeddings('train', embeddings_dir)
X_test, y_test   = load_embeddings('test',  embeddings_dir)

print(f'Train X: {X_train.shape} | dtype: {X_train.dtype}')
print(f'Test  X: {X_test.shape}  | dtype: {X_test.dtype}')
print(f'Rango de valores — min: {X_train.min():.3f} | max: {X_train.max():.3f}')